In [0]:


 
-- ,
--      coalesce(dissolution_date < to_date(left(_last_seen_snapshot, 8), 'yyyyMMdd'), false) AS is_dissolved        
--  array_contains(topics,'sanction') AS is_sanctioned

-- SELECT canonical_id, name FROM A

MERGE INTO pep_dev.silver.legal_entity AS t 
USING(

WITH A AS (
SELECT  
    psr.canonical_id,
    MAX(CASE WHEN prop = 'name' THEN value END) AS name, --name
    coalesce(
            CASE
            WHEN array_contains(collect_set(schema), 'Company') THEN 'Company' END,
            CASE
            WHEN array_contains(collect_set(schema),'PublicBody') THEN 'PublicBody' END, 
            CASE
            WHEN array_contains(collect_set(schema),'Organization') THEN 'Organization'END,
            CASE
            WHEN array_contains(collect_set(schema),'LegalEntity') THEN 'LegalEntity' END
            ) AS entity_type,
    collect_set(value)  FILTER( WHERE prop IN ('name', 'alias', 'previousName', 'weakAlias')) AS alt_names,
    
    coalesce (
        MAX(CASE WHEN prop = 'jurisdiction' THEN value END),
        MAX(CASE WHEN prop = 'mainCountry' THEN value END),
        MAX(CASE WHEN prop = 'country' THEN value END)
    ) AS country,

    MAX(CASE WHEN prop = 'legalForm' THEN value END) AS legal_form,
    try_cast(MIN(CASE WHEN prop='incorporationDate' THEN value END) AS DATE) AS incorporation_date,
    try_cast(MAX(CASE WHEN prop='dissolutionDate' THEN VALUE END) AS DATE) AS  dissolution_date,
    collect_set(value) FILTER(WHERE prop = 'innCode') AS inn_codes ,
    collect_set(value) FILTER(WHERE prop = 'ogrnCode') AS ogrn_codes,
    collect_set(value) FILTER(WHERE prop = 'registrationNumber') AS registration_numbers,
    collect_set(value) FILTER(WHERE prop = 'taxNumber') AS tax_numbers,
    collect_set(value) FILTER(WHERE prop = 'leiCode') AS lei_codes,
    collect_set(value) FILTER(WHERE prop = 'topics') AS topics,
   
    collect_set(dataset) AS datasets,
    MIN(_snapshot_version) AS _first_seen_snapshot  ,
    MAX(_snapshot_version) AS _last_seen_snapshot
    
FROM 
pep_dev.bronze.statements AS pbs
JOIN pep_dev.silver.referent AS psr
ON psr.entity_id = pbs.entity_id  
WHERE SCHEMA IN ('Company','Organization','LegalEntity','PublicBody')
AND psr.canonical_id NOT IN (SELECT canonical_id FROM pep_dev.silver.referent WHERE referent_kind = 'Person')
AND _snapshot_version=(
SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
-- AND prop IN ('name','alias','previousName','weakAlias','country','jurisdiction','mainCountry',
-- 'legalForm','topics','incorporationDate','dissolutionDate','innCode','ogrnCode','registrationNumber','taxNumber','leiCode')
GROUP BY psr.canonical_id

-- LIMIT 1000

)


SELECT   A.*, 
coalesce(dissolution_date < to_date(left(_last_seen_snapshot, 8), 'yyyyMMdd'), false) AS is_dissolved,
array_contains(topics,'sanction') AS is_sanctioned FROM A
) AS s
ON 

t.canonical_id = s.canonical_id

WHEN MATCHED THEN UPDATE
SET 
    t.name=s.name
    ,t.entity_type=s.entity_type
    ,t.alt_names=s.alt_names
    ,t.country=s.country
    ,t.legal_form=s.legal_form
    ,t.incorporation_date=s.incorporation_date
    ,t.dissolution_date=s.dissolution_date
    ,t.inn_codes=s.inn_codes
    ,t.ogrn_codes=s.ogrn_codes
    ,t.registration_numbers=s.registration_numbers
    ,t.tax_numbers=s.tax_numbers
    ,t.lei_codes=s.lei_codes
    ,t.topics=s.topics
    ,t.datasets=s.datasets
    ,t._last_seen_snapshot=s._last_seen_snapshot
    ,t.is_dissolved=s.is_dissolved
    ,t.is_sanctioned=s.is_sanctioned

WHEN NOT MATCHED THEN INSERT (
    canonical_id,
    name,
    entity_type,
    alt_names,
    country,
    legal_form,
    incorporation_date,
    dissolution_date,
    inn_codes,
    ogrn_codes,
    registration_numbers,
    tax_numbers,
    lei_codes,
    topics,
    datasets,
    _first_seen_snapshot,
    _last_seen_snapshot,
    is_dissolved,
    is_sanctioned
) VALUES (
    s.canonical_id,
    s.name,
    s.entity_type,
    s.alt_names,
    s.country,
    s.legal_form,
    s.incorporation_date,
    s.dissolution_date,
    s.inn_codes,
    s.ogrn_codes,
    s.registration_numbers,
    s.tax_numbers,
    s.lei_codes,
    s.topics,
    s.datasets,
    s._first_seen_snapshot,
    s._last_seen_snapshot,
    s.is_dissolved,
    s.is_sanctioned
)

In [0]:
-- SELECT count(*), count(DISTINCT canonical_id) FROM pep_dev.silver.legal_entity